## Crypto Coins Selected: 
- Bitcoin, Ethereum, Solana, Dogecoin, Tether

In [ ]:
import requests
import pandas as pd
import os 
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

url = "https://api.coinpaprika.com/v1/tickers?quotes=USD"

response = requests.get(url)

print(response.text)

response.json()

In [ ]:
crypto_data = response.json()

In [ ]:
crypto_df = pd.DataFrame(crypto_data)

crypto_df.head()


In [ ]:
my_coins = ['btc-bitcoin', 'eth-ethereum', 'sol-solana', 'doge-dogecoin', 'usdt-tether']

coins_df = crypto_df[crypto_df['id'].isin(my_coins)].copy()

coins_df


In [ ]:
# Creating new columns and extracting the price, volume and market cap

coins_df['price'] = [row['USD']['price'] for row in coins_df['quotes']]

coins_df['volume'] = [row['USD']['volume_24h'] for row in coins_df['quotes']]

coins_df['market_cap'] = [row['USD']['market_cap'] for row in coins_df['quotes']]


coins_df[['id', 'name', 'price', 'volume', 'market_cap']]



In [ ]:
# Filter and clean columns

useful_columns = ['id', 'name', 'symbol', 'rank', 'price', 'volume', 'market_cap']

crypto_df = coins_df[useful_columns].copy()

crypto_df = crypto_df.rename(columns={'id': 'coin_id'})

crypto_df


In [ ]:
# shows the currect UTC time
crypto_df['extracted_at'] = pd.Timestamp.now('UTC')

# shows how active the trading is relative to the coin's size
crypto_df['vol_to_market_cap_ratio'] = crypto_df['volume'] / crypto_df['market_cap']

crypto_df

In [ ]:
# store to db

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

crypto_df.to_sql('crypto_data', con=engine, if_exists='replace', index=False)

